## Dependencies and setup

In [1]:
import os
import torch
import wandb
from PIL import Image
from ultralytics import YOLO

In [2]:
from dotenv import load_dotenv

RUN_PATH = os.getenv("RUN_PATH")
MODEL_WEIGHTS_PATH = os.getenv("MODEL_WEIGHTS_PATH")
DATASETS_PATH = os.getenv("IMAGE_PATH")
WANDB_KEY = os.getenv("WANDB_KEY")

In [3]:
from ultralytics import settings

settings.update(
    {
        "tensorboard": False,
        "wandb": True
    }
)

## Training

In [4]:
dataset_path = os.path.join(DATASETS_PATH, "mvtec_yolo_test", "fold_1")

In [5]:
model_checkpoint_name = "rtdetr-l.pt"

In [6]:
augmentation_zero = {
    "hsv_h": 0.0, # Modifier la teinte de l'image
    "hsv_s": 0.0, # Modifier la saturation de l'image
    "hsv_v": 0.0, # Modifier la luminosité
    "degrees": 0,
    "translate": 0.0,
    "scale": 0.0,
    "shear": 0.0,
    "perspective": 0.0, # Utile pour la compréhension 3D
    "flipud": 0.0, # Vertical flip prob
    "fliplr": 0.0, # Horizontal flip prob
    "bgr": 0.0, # Shuffle channels prob
    "mosaic": 0.0, # 4 images en 1
    "mixup": 0.0, # Superposition de 2 images, en jouant sur la transparence
    "cutmix": 0.0, # Crop une partie d'une image et la coller sur une autre
    "erasing": 0.0 # Efface une région aléatoire de l'image
}

augmentation_light = {
    "hsv_h": 0.015, # Modifier la teinte de l'image
    "hsv_s": 0.4, # Modifier la saturation de l'image
    "hsv_v": 0.4, # Modifier la luminosité
    "degrees": 0,
    "translate": 0.0,
    "scale": 0.0,
    "shear": 0.0,
    "perspective": 0.0, # Utile pour la compréhension 3D
    "flipud": 0.2, # Vertical flip prob
    "fliplr": 0.2, # Horizontal flip prob
    "bgr": 0.0, # Shuffle channels prob
    "mosaic": 0.0, # 4 images en 1
    "mixup": 0.0, # Superposition de 2 images, en jouant sur la transparence
    "cutmix": 0.0, # Crop une partie d'une image et la coller sur une autre
    "erasing": 0.0 # Efface une région aléatoire de l'image
}

augmentation_medium = {
    "hsv_h": 0.5, # Modifier la teinte de l'image
    "hsv_s": 0.8, # Modifier la saturation de l'image
    "hsv_v": 0.8, # Modifier la luminosité
    "degrees": 0,
    "translate": 0.0,
    "scale": 0.0,
    "shear": 0.0,
    "perspective": 0.0, # Utile pour la compréhension 3D
    "flipud": 0.5, # Vertical flip prob
    "fliplr": 0.5, # Horizontal flip prob
    "bgr": 0.2, # Shuffle channels prob
    "mosaic": 0.0, # 4 images en 1
    "mixup": 0.0, # Superposition de 2 images, en jouant sur la transparence
    "cutmix": 0.3, # Crop une partie d'une image et la coller sur une autre
    "erasing": 0.2 # Efface une région aléatoire de l'image
}

hyperparameters = {
    "freeze": 10,
    "conf": 0.001,
    "patience": 20,
}

In [7]:
if os.path.exists(os.path.join(MODEL_WEIGHTS_PATH, model_checkpoint_name)):
    model = YOLO(os.path.join(MODEL_WEIGHTS_PATH, model_checkpoint_name))
else:
    model = YOLO(model_checkpoint_name)

# model = YOLO("/Users/theo.moreau/Documents/futur/src/od_training/Object_detection/train38/weights/last.pt")

wandb.login(key=WANDB_KEY)

# Train the model on the COCO8 example dataset for 100 epochs
results = model.train(
                data=os.path.join(dataset_path, "mvtec.yaml"), 
                epochs=50, 
                imgsz=640, 
                device="mps", 
                project="Object_detection",
                **augmentation_medium,
                **hyperparameters
            )

100%|██████████| 63.4M/63.4M [00:03<00:00, 19.4MB/s]
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/theo.moreau/.netrc
wandb: Currently logged in as: theomoreau-thor (theomoreau-thor-octo-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


New https://pypi.org/project/ultralytics/8.3.144 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.141 🚀 Python-3.9.22 torch-2.1.2 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.2, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.3, data=/Users/theo.moreau/Documents/futur/datasets/mvtec_yolo_test/fold_1/mvtec.yaml, degrees=0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.2, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.5, hsv_s=0.8, hsv_v=0.8, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=0.0, multi_scale=False, name=train45, nbs=64, nms=False,

Freezing layer 'model.0.stem1.conv.weight'
Freezing layer 'model.0.stem1.bn.weight'
Freezing layer 'model.0.stem1.bn.bias'
Freezing layer 'model.0.stem2a.conv.weight'
Freezing layer 'model.0.stem2a.bn.weight'
Freezing layer 'model.0.stem2a.bn.bias'
Freezing layer 'model.0.stem2b.conv.weight'
Freezing layer 'model.0.stem2b.bn.weight'
Freezing layer 'model.0.stem2b.bn.bias'
Freezing layer 'model.0.stem3.conv.weight'
Freezing layer 'model.0.stem3.bn.weight'
Freezing layer 'model.0.stem3.bn.bias'
Freezing layer 'model.0.stem4.conv.weight'
Freezing layer 'model.0.stem4.bn.weight'
Freezing layer 'model.0.stem4.bn.bias'
Freezing layer 'model.1.m.0.conv.weight'
Freezing layer 'model.1.m.0.bn.weight'
Freezing layer 'model.1.m.0.bn.bias'
Freezing layer 'model.1.m.1.conv.weight'
Freezing layer 'model.1.m.1.bn.weight'
Freezing layer 'model.1.m.1.bn.bias'
Freezing layer 'model.1.m.2.conv.weight'
Freezing layer 'model.1.m.2.bn.weight'
Freezing layer 'model.1.m.2.bn.bias'
Freezing layer 'model.1.m.3.

train: Scanning /Users/theo.moreau/Documents/futur/datasets/mvtec_yolo_test/fold_1/labels/train.cache... 355 images, 262 backgrounds, 0 corrupt: 100%|██████████| 355/355 [00:00<?, ?it/s]

val: Fast image access ✅ (ping: 0.1±0.1 ms, read: 399.8±100.4 MB/s, size: 396.4 KB)


...

                   all         95         30      0.845      0.726      0.753      0.351
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 22, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



42 epochs completed in 6.518 hours.
Optimizer stripped from Object_detection/train45/weights/last.pt, 66.1MB
Optimizer stripped from Object_detection/train45/weights/best.pt, 66.1MB

Validating Object_detection/train45/weights/best.pt...
Ultralytics 8.3.141 🚀 Python-3.9.22 torch-2.1.2 MPS (Apple M4)
rt-detr-l summary: 302 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:26<00:00,  8.85s/it]


                   all         95         30      0.943      0.767      0.808      0.413
Speed: 1.1ms preprocess, 249.6ms inference, 0.0ms loss, 8.0ms postprocess per image
Results saved to Object_detection/train45


lr/pg0,▁▃▅▇███▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁
lr/pg1,▁▃▅▇███▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁
lr/pg2,▁▃▅▇███▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁
metrics/mAP50(B),▁▄▄▅▄▅▄▄▅▆▆▇▆▆▇▆▇▇▆▆██▇▇█▇█▇▇▇▇▇█▇███▇▇█
metrics/mAP50-95(B),▁▄▄▄▃▅▄▄▆▆▅▇▆▇▇▆▆▇▆▆██▆▇▇▇▇▇▇▇▇▇▇▇▇▇█▇▇█
metrics/precision(B),▁▇▅▆▆▆▆▆▅▅▆▇▆▆▇▇▇▇▅███▇▇██▇▇▇▇▇▇█▇▇█▇▇▇█
metrics/recall(B),▂▁▃▄▂▅▂▄▆▅▇▇▅▆▅▇▆▅▇▄█▇▇▆▇▇▇▇▇▆▇▇▇▆██▇▇▇█
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
train/cls_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
